# General guidelines

- Refer to the notebook of the tutorial session taken on 15th October for implementation of following problems
- Download a copy of this notebook and perform your implementations here.
- Submit this notebook after implementation using the following format: firstname_srNO(last 5 digits).pynb
eg. mahesh_20369.ipynb

# Q1

## Objective


In this assignment, you will design and implement a lightweight LLM-powered agent that interacts with an Airbnb-like server API using structured data models.
Your agent will be responsible for verifying the validity of links returned by the Airbnb endpoints and ensuring that only working URLs appear in the final output.



## Q1 Guidelines

All students must use a common lightweight language model to ensure fairness in evaluation (e.g., gpt-4o-mini, Mistral-7B-Instruct, Phi-3-mini, or Llama 3 8B Instruct).
You may, however, experiment with:

- System prompts

- Verifier agent logic

### Functional Requirements

You are required to implement three functions that interact with the Airbnb server:

- get_current_time(params: dict)
Returns the current time from the Airbnb server.

- airbnb_search(params: dict)
Performs a search query (e.g., by location, check-in date, price range) on the Airbnb server.

- airbnb_listing_details(params: dict)
Fetches detailed information for a specific listing.

Each function must produce structured, validated output using Pydantic models.

### Define Structured Data Models

Use Pydantic to define schemas for:

- Current time response

- Search results (with listing summaries)

- Listing details

Each schema should validate key fields such as IDs, titles, prices, amenities, and URLs.

### Implement Airbnb API Wrappers

Each of the three required functions will call the respective Airbnb API endpoints and return JSON responses that are parsed and validated against your Pydantic models.
You may connect to a provided Airbnb mock server or simulate your own endpoints.

### Recommended Libraries

You may use the following Python libraries:

- Pydantic → Schema validation

- Requests → HTTP calls and link verification

- Tenacity → Retry mechanism for failed link checks

- Transformers or OpenAI API → Lightweight LLM interface


FastAPI / Flask (optional) → To simulate a local Airbnb mock API









In [1]:
# you should dedicate a code cell next to the markdown cell "OUTPUT" to produce a final JSON output of the following form



{
  "current_time": "YYYY-MM-DDTHH:MM:SSZ",
  "search_results": [
    {
      "id": "A101",
      "title": "Cozy apartment in Paris",
      "url": "https://www.airbnb.com/rooms/12345",
      "price_per_night": 120.0
    }
  ],
  "verified_links": [
    "https://www.airbnb.com/rooms/12345"
  ]
}




{'current_time': 'YYYY-MM-DDTHH:MM:SSZ',
 'search_results': [{'id': 'A101',
   'title': 'Cozy apartment in Paris',
   'url': 'https://www.airbnb.com/rooms/12345',
   'price_per_night': 120.0}],
 'verified_links': ['https://www.airbnb.com/rooms/12345']}

In [2]:
# !curl -fsSL https://ollama.com/install.sh | sh
%pip install pydantic tenacity requests fastapi uvicorn


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import subprocess
import time

# Start Ollama server in background
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait a moment so the server boots
time.sleep(5)

print("✅ Ollama server started in background")

✅ Ollama server started in background


In [4]:
# Importing this module starts the FastAPI mock server in a background thread
import importlib
import mock_airbnb_server  # noqa: F401

importlib.reload(mock_airbnb_server)

BASE_URL = "http://127.0.0.1:8001"
print("Mock Airbnb server should be running at", BASE_URL)


Mock Airbnb API server started on http://127.0.0.1:8001 🚀
Mock Airbnb API server started on http://127.0.0.1:8001 🚀
Mock Airbnb server should be running at http://127.0.0.1:8001


In [5]:
from datetime import datetime
from typing import List

from pydantic import BaseModel, HttpUrl, field_validator


class CurrentTimeResponse(BaseModel):
    current_time: datetime


class ListingSummary(BaseModel):
    id: str
    title: str
    url: HttpUrl
    price_per_night: float


class SearchResultsResponse(BaseModel):
    results: List[ListingSummary]


class ListingDetails(BaseModel):
    id: str
    title: str
    price_per_night: float
    amenities: List[str]
    description: str
    url: HttpUrl


class LinkVerification(BaseModel):
    id: str
    url: HttpUrl
    is_working: bool
    reason: str

    @field_validator("reason")
    @classmethod
    def non_empty_reason(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("reason must not be empty")
        return v


class FinalOutput(BaseModel):
    current_time: datetime
    search_results: List[ListingSummary]
    verified_links: List[HttpUrl]


In [6]:
import requests

def safe_json(resp):
    try:
        return resp.json()
    except Exception:
        raise ValueError(
            f"Expected JSON but got:\nStatus={resp.status_code}\nBody:\n{resp.text[:500]}"
        )


def get_current_time(params=None):
    url = f"{BASE_URL}/current_time"
    resp = requests.get(url, params=params or {})
    resp.raise_for_status()
    return CurrentTimeResponse(**safe_json(resp))


def airbnb_search(params):
    url = f"{BASE_URL}/search"
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    return SearchResultsResponse(**safe_json(resp))


def airbnb_listing_details(params):
    listing_id = params["listing_id"]
    url = f"{BASE_URL}/listing/{listing_id}"
    resp = requests.get(url)
    resp.raise_for_status()
    return ListingDetails(**safe_json(resp))


In [ ]:
import json
from typing import Optional


OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3:8b"  # Change if you use another lightweight model (mistral, phi3, etc.)


def ollama_chat(
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0.2,
    json_mode: bool = False,
) -> str:
    """
    Minimal helper to call Ollama's /api/chat endpoint.
    """
    payload: dict = {
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "stream": False,
    }

    # if json_mode:
    #     # Instruct Ollama to return JSON only
    #     payload["format"] = "json"

    resp = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    # Ollama streams by default; but when using api/chat without "stream": true,
    # we get the final message content in data["message"]["content"].
    content: str = data["message"]["content"]
    return content.strip()


In [ ]:
from tenacity import retry, stop_after_attempt, wait_fixed


@retry(stop=stop_after_attempt(3), wait=wait_fixed(0.5))
def http_check_url(url: str) -> tuple[int, str]:
    """
    Simple HTTP GET to check if a URL is working.
    Returns (status_code, body_snippet)
    """
    resp = requests.get(url, timeout=5)
    status = resp.status_code
    text = resp.text[:500] if resp.text else ""
    return status, text


def verify_listing_links(listings: list[ListingSummary]) -> list[LinkVerification]:
    """
    Uses HTTP checks + an Ollama LLM to decide if each link is working.
    """
    system_prompt = (
        "You are a strict link-verification agent. "
        "You will be given metadata about an Airbnb listing, "
        "the HTTP status code when fetching its URL, and a short snippet of the page.\n\n"
        "Return a JSON object with keys: id, url, is_working (boolean), reason (string).\n"
        "Rules:\n"
        "- Consider a link working only if status code is 200 and the page content matches the listing.\n"
        "- If status is not 200 or the content seems unrelated, mark is_working as false.\n"
        "- Keep reason short but specific.\n"
        "- Do not invent new URLs."
    )

    verifications: list[LinkVerification] = []

    for listing in listings:
        try:
            status_code, snippet = http_check_url(str(listing.url))
        except Exception as e:  # network failure, timeout, etc.
            status_code, snippet = 0, f"Request failed with error: {e}"

        user_payload = {
            "listing": {
                "id": listing.id,
                "title": listing.title,
                "url": str(listing.url),
                "price_per_night": listing.price_per_night,
            },
            "http_check": {
                "status_code": status_code,
                "body_snippet": snippet,
            },
        }

        user_prompt = (
            "Here is the data:\n"
            + json.dumps(user_payload, indent=2)
            + "\n\n"
            "Decide if the link is working. "
            "Respond with a single JSON object like:\n"
            '{"id": "...", "url": "...", "is_working": true/false, "reason": "..."}'
        )

        raw = ollama_chat(system_prompt, user_prompt, temperature=0.0)


        try:
            cleaned = extract_json(raw)
            verification = LinkVerification.model_validate_json(cleaned)
        except Exception:
            print("Failed to parse LLM output, falling back to basic HTTP check.")
            
            verification = LinkVerification(
                id=listing.id,
                url=listing.url,
                is_working=status_code == 200,
                reason=f"Fallback decision based on HTTP status {status_code}.",
            )

        verifications.append(verification)

    return verifications

import re

def extract_json(text: str) -> str:
    """
    Extract JSON object from an LLM response.
    Handles extra text, markdown, or explanations.
    """
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in LLM output.")
    return match.group(0)



## Evaluation Criteria

| **Component**                      | **Points** | **Description**                                                      |
| ---------------------------------- | ---------- | -------------------------------------------------------------------- |
| Pydantic model definitions         | 20         | Models correctly represent response structure and enforce validation |
| Airbnb API wrapper functions       | 20         | Functions correctly fetch and return structured data                 |
| Verifier agent implementation      | 25         | Accurately checks and filters out invalid URLs                       |
| Integration with LLM               | 15         | LLM used effectively for reasoning or verification                   |
| Output format and reproducibility  | 10         | JSON output correctly structured and reproducible                    |
| Prompt design and agent creativity | 10         | Quality and clarity of prompt/system design                          |


## Output

In [13]:
# use this cell to generate output for Q1.
from pathlib import Path


# 1. Get current time from server
current_time_response = get_current_time({})
print("Current time response:", current_time_response)

# 2. Run a search on the Airbnb mock server
search_params = {
    "location": "Berlin",   # match broad subset of titles (Paris, Berlin, Goa)
    "min_price": 0,
    "max_price": 1000,
}
search_results = airbnb_search(search_params)
print("Search results:", search_results)

# 3. Verify each listing's URL using LLM-based agent
verifications = verify_listing_links(search_results.results)

# 4. Filter to only working links
verified_links = [v.url for v in verifications if v.is_working]

# 5. Construct final structured output
final_output = FinalOutput(
    current_time=current_time_response.current_time,
    search_results=search_results.results,
    verified_links=verified_links,
)

# 6. Convert to the exact JSON format required by the assignment
output_dict = {
    "current_time": final_output.current_time.isoformat() + "Z",
    "search_results": [
        {
            "id": listing.id,
            "title": listing.title,
            "url": str(listing.url),
            "price_per_night": float(listing.price_per_night),
        }
        for listing in final_output.search_results
    ],
    "verified_links": [str(url) for url in final_output.verified_links],
}

# Pretty print the JSON
print("=== Final JSON output for Q1 ===")
print(json.dumps(output_dict, indent=2))

# Save to disk so Q2 can read it
output_path = Path("verified_output_q1.json")
output_path.write_text(json.dumps(output_dict, indent=2), encoding="utf-8")
print(f"\nSaved verified JSON to {output_path.resolve()}")


INFO:     127.0.0.1:62374 - "GET /current_time HTTP/1.1" 200 OK
Current time response: current_time=datetime.datetime(2025, 11, 30, 7, 40, 51, 614630)
INFO:     127.0.0.1:62375 - "GET /search?location=Berlin&min_price=0&max_price=1000 HTTP/1.1" 200 OK
Search results: results=[ListingSummary(id='A102', title='Modern loft in Berlin', url=HttpUrl('http://127.0.0.1:8001/rooms/A102'), price_per_night=90.0)]
INFO:     127.0.0.1:62376 - "GET /rooms/A102 HTTP/1.1" 200 OK
=== Final JSON output for Q1 ===
{
  "current_time": "2025-11-30T07:40:51.614630Z",
  "search_results": [
    {
      "id": "A102",
      "title": "Modern loft in Berlin",
      "url": "http://127.0.0.1:8001/rooms/A102",
      "price_per_night": 90.0
    }
  ],
  "verified_links": [
    "http://127.0.0.1:8001/rooms/A102"
  ]
}

Saved verified JSON to D:\Gen AI\verified_output_q1.json


# Q2

## Objective

Using the verified JSON output generated in Question 1, design an LLM-based module that parses and reformats the data into a clean, human-readable list of Airbnb listings.
Unlike Question 1, this stage does not perform link verification — it assumes all links are already valid.

Your goal is to:

- Feed the verified JSON back to the LLM,

- Prompt it to extract only the key user-facing details, and

- Produce a clean, structured list or summary that can be directly displayed to end users or used by downstream systems


## Q2 Guidelines


### 1. Load the Verified JSON

Load the JSON object produced in Question 1 into your notebook.
Ensure your code dynamically reads the keys rather than hard-coding them.

### 2. Design an LLM Prompt

Construct a prompt that:

- Accepts the JSON as context.

- Instructs the model to return a clean, concise list summarizing each listing’s:

  - Title

  - Price per night

  - URL

- Optionally, includes additional natural-language formatting (e.g., numbered list, short summary lines).

- Explicitly forbids re-verifying or modifying URLs.


### 3. Ensure the LLM response:

Matches all entries in the verified list.

Contains no hallucinated or missing items.

Requires no further link verification.








## Evaluation Criteria

| **Component**                         | **Points** | **Description**                                                      |
| ------------------------------------- | ---------- | -------------------------------------------------------------------- |
| JSON ingestion & preprocessing        | 10         | Correctly loads and handles verified JSON                            |
| Prompt construction                   | 10         | Clear and robust system/user prompt design                           |
| Clean, correctly formatted LLM output | 20         | Output includes all valid listings, no verification or hallucination |
| Reproducibility & LLM configuration   | 10         | Uses consistent lightweight model with documented parameters         |




In [14]:
from pathlib import Path
import json

verified_path = Path("verified_output_q1.json")
if not verified_path.exists():
    raise FileNotFoundError("verified_output_q1.json not found. Run the Q1 cell first.")

verified_json = json.loads(verified_path.read_text(encoding="utf-8"))

print("Keys in verified JSON:", list(verified_json.keys()))
print("Number of verified links:", len(verified_json.get("verified_links", [])))


Keys in verified JSON: ['current_time', 'search_results', 'verified_links']
Number of verified links: 1


In [15]:
def render_listings_for_users(verified_data: dict) -> str:
    """
    Uses Ollama to turn the verified JSON into a clean user-facing list.
    No link verification or URL changes happen here.
    """
    system_prompt = (
        "You are a helpful assistant that formats Airbnb search results for end users.\n"
        "You will be given a JSON object produced by a previous verification step.\n\n"
        "Important rules:\n"
        "- Assume that every URL in `verified_links` is already valid.\n"
        "- Do NOT modify, filter, or re-verify URLs.\n"
        "- Do NOT invent new listings or URLs.\n"
        "- For each listing that appears in `search_results` AND whose URL "
        "is present in `verified_links`, output a clean, human-readable summary.\n"
        "- For each listing, include:\n"
        "  * A numbered index\n"
        "  * Title\n"
        "  * Price per night\n"
        "  * URL\n"
        "- Output in plain text, for example a numbered list."
    )

    user_prompt = (
        "Here is the verified JSON that you must use:\n\n"
        + json.dumps(verified_data, indent=2)
        + "\n\n"
        "Produce a numbered list of all listings whose URL is in `verified_links`.\n"
        "Each line should look like:\n"
        "1. <Title> - $<price_per_night>/night - <URL>\n"
        "Do not skip any verified listing and do not add anything extra."
    )

    formatted = ollama_chat(system_prompt, user_prompt, temperature=0.0, json_mode=False)
    return formatted


In [16]:
user_facing_output = render_listings_for_users(verified_json)

print("=== Final formatted listing output for Q2 ===\n")
print(user_facing_output)


=== Final formatted listing output for Q2 ===

Here is the output:

1. Modern loft in Berlin - $90/night - http://127.0.0.1:8001/rooms/A102


INFO:     127.0.0.1:62369 - "GET /rooms/A102 HTTP/1.1" 200 OK
